# Task 2 (part 1) — Tiers A & B + Leakage Audit

Two detectors on the **shared frozen split** from Task 0, structured as an ablation (not a
leaderboard).

In [1]:
%pip install -q xgboost scikit-learn gensim numpy pandas torch

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [2]:
DATA = Path("data/dataset")
def load_jsonl(p): return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]
train, val, test = (pd.DataFrame(load_jsonl(DATA/f"{s}.jsonl")) for s in ("train","val","test"))
print({s: len(d) for s,d in [("train",train),("val",val),("test",test)]})

{'train': 882, 'val': 189, 'test': 189}


## 0. Leakage audit
(1) no text appears in two splits; (2) per-class test length distributions are ~equal;
(3) no class-exclusive vocabulary dominates; (4) a length-only classifier should be near chance.

In [ ]:
def audit(train, val, test):
    # (1) overlap
    for a,b in [("train",train),("val",val),("test",test)]:
        pass
    sets = {n: set(d["text"]) for n,d in [("train",train),("val",val),("test",test)]}
    print("overlap train/test:", len(sets["train"] & sets["test"]),
          "| train/val:", len(sets["train"] & sets["val"]),
          "| val/test:", len(sets["val"] & sets["test"]), "(all must be 0)")
    # (2) length per class on test
    print("\ntest mean word_count by class:")
    print(test.groupby("label")["word_count"].agg(["mean","std"]).round(1))
    # (4) length-only ablation
    from sklearn.tree import DecisionTreeClassifier
    Ltr = train["word_count"].values.reshape(-1,1); Lte = test["word_count"].values.reshape(-1,1)
    acc = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Ltr, train["class_id"]).score(Lte, test["class_id"])
    print(f"\nlength-only classifier test acc = {acc:.3f}  (≈0.33 is good; high => length leaks)")
audit(train, val, test)

overlap train/test: 0 | train/val: 0 | val/test: 0 (all must be 0)

test mean word_count by class:
             mean   std
label                  
ai_neutral  165.1  13.6
ai_styled   145.6   7.8
human       154.9  44.0

length-only classifier test acc = 0.709  (≈0.33 is good; high => length leaks)


## 1. Tier A — Statistician (XGBoost on Task-1 features)
Reuses the stylometric features from Task 1.

In [ ]:
import xgboost as xgb

import re
from collections import Counter
WORD = re.compile(r"[a-zA-Z']+")
def feat_row(t):
    w = WORD.findall(t.lower()); n = max(len(w)/1000,1e-6)
    sents=[s for s in re.split(r"[.!?]+",t) if s.strip()]; sl=[len(s.split()) for s in sents]
    c = Counter(w[:120]) if len(w)>=120 else Counter(w)
    return {"sent_var": float(np.std(sl)) if len(sl)>1 else 0.0,
            "hapax": (sum(1 for x in c.values() if x==1)/len(c)) if c else 0,
            "ttr": (len(set(w[:120]))/120) if len(w)>=120 else len(set(w))/max(len(w),1),
            "semicolon": t.count(";")/n, "emdash": (t.count("\u2014")+t.count("--"))/n,
            "comma": t.count(",")/n, "exclaim": t.count("!")/n}
def Xy(d):
    X = pd.DataFrame([feat_row(t) for t in d["text"]]).values
    return X, d["class_id"].values
Xtr,ytr = Xy(train); Xte,yte = Xy(test)

In [5]:
def run_tier_a(ytr, yte, mapping=None, name=""):
    yt = np.array([mapping[c] for c in ytr]) if mapping else ytr
    ye = np.array([mapping[c] for c in yte]) if mapping else yte
    keep_tr = yt >= 0; keep_te = ye >= 0
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                            eval_metric="mlogloss", random_state=42)
    clf.fit(Xtr[keep_tr], yt[keep_tr])
    p = clf.predict(Xte[keep_te])
    acc = accuracy_score(ye[keep_te], p); f1 = f1_score(ye[keep_te], p, average="macro")
    print(f"[Tier A · {name}] acc={acc:.3f} f1={f1:.3f}")
    return clf, acc, f1

In [6]:
# class_id: 0=human 1=ai_neutral 2=ai_styled
print("Tertiary (3-way):"); run_tier_a(ytr, yte, None, "tertiary")
print("Binary framings:")
run_tier_a(ytr, yte, {0:0,1:1,2:1}, "human-vs-AI")
run_tier_a(ytr, yte, {0:-1,1:0,2:1}, "neutral-vs-styled (hard)")
run_tier_a(ytr, yte, {0:0,1:-1,2:1}, "human-vs-styled (imposter)")

Tertiary (3-way):
[Tier A · tertiary] acc=0.841 f1=0.841
Binary framings:
[Tier A · human-vs-AI] acc=0.968 f1=0.964
[Tier A · neutral-vs-styled (hard)] acc=0.802 f1=0.801
[Tier A · human-vs-styled (imposter)] acc=0.952 f1=0.952


(XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='mlogloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.1, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=4, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=200, n_jobs=None,
               num_parallel_tree=None, ...),
 0.9523809523809523,
 0.9523809523809523)

In [ ]:
clf,_,_ = run_tier_a(ytr, yte, None, "tertiary(for importances)")
cols = list(feat_row(train["text"].iloc[0]).keys())
print("\nfeature importances:", dict(zip(cols, clf.feature_importances_.round(3))))

[Tier A · tertiary(for importances)] acc=0.841 f1=0.841

feature importances: {'sent_var': np.float32(0.07), 'hapax': np.float32(0.049), 'ttr': np.float32(0.064), 'semicolon': np.float32(0.592), 'emdash': np.float32(0.205), 'comma': np.float32(0.019), 'exclaim': np.float32(0.0)}


## 2. Tier B — Semanticist (FFNN on averaged GloVe embeddings)

In [ ]:
import gensim.downloader as api
glove = api.load("glove-wiki-gigaword-100")    # ~130MB first time

In [9]:
def embed(text):
    vecs = [glove[w] for w in WORD.findall(text.lower()) if w in glove]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

In [10]:
Etr = np.vstack([embed(t) for t in train["text"]])
Ete = np.vstack([embed(t) for t in test["text"]])

In [11]:
import torch, torch.nn as nn
class MLP(nn.Module):
    def __init__(self, d_in=100, d_h=128, n_cls=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in,d_h), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(d_h,n_cls))
    def forward(self,x): return self.net(x)

In [12]:
def run_tier_b(ytr, yte, n_cls, name=""):
    Xt = torch.tensor(Etr, dtype=torch.float32); Yt = torch.tensor(ytr, dtype=torch.long)
    Xe = torch.tensor(Ete, dtype=torch.float32)
    m = MLP(n_cls=n_cls); opt = torch.optim.Adam(m.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss()
    for _ in range(150):
        opt.zero_grad(); loss = lossf(m(Xt), Yt); loss.backward(); opt.step()
    p = m(Xe).argmax(1).numpy()
    acc = accuracy_score(yte, p); f1 = f1_score(yte, p, average="macro")
    print(f"[Tier B · {name}] acc={acc:.3f} f1={f1:.3f}")
    return m

In [13]:
run_tier_b(ytr, yte, 3, "tertiary")
# Binary human-vs-AI:
mp = {0:0,1:1,2:1}
run_tier_b(np.array([mp[c] for c in ytr]), np.array([mp[c] for c in yte]), 2, "human-vs-AI")

[Tier B · tertiary] acc=0.995 f1=0.995
[Tier B · human-vs-AI] acc=0.995 f1=0.994


MLP(
  (net): Sequential(
    (0): Linear(in_features=100, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import torch
m = run_tier_b(ytr, yte, 3, "tertiary")
p = m(torch.tensor(Ete, dtype=torch.float32)).argmax(1).numpy()
print(confusion_matrix(yte, p))

probe = [json.loads(l) for l in open("data/dataset/probe_modern_human.jsonl")]
Ep = np.vstack([embed(r["text"]) for r in probe])
pp = m(torch.tensor(Ep, dtype=torch.float32)).argmax(1).numpy()
print("modern-human predicted as class:", np.bincount(pp))

[Tier B · tertiary] acc=0.995 f1=0.995
[[62  0  1]
 [ 0 63  0]
 [ 0  0 63]]
modern-human predicted as class: [51  0  9]
